# LoRA vs KronA — fast Unsloth fine-tune

Same experiment as before — fine-tune **twice** (once with **LoRA**, once with **KronA**),
then compare — but now it runs on **Unsloth**:

- **Unsloth** loads and patches the model, so we get its fast kernels, 4-bit loading and
  memory-safe gradient checkpointing. No more 8.75 GiB fp32 upcast OOM.
- The model is **smaller and text-only** (`unsloth/gemma-3-1b-it`, 4-bit) so a whole
  LoRA-vs-KronA run finishes in a few minutes on a free T4.
- **Fairness is kept**: both adapters are attached with plain `peft.get_peft_model()` on the
  *same* Unsloth-loaded base, so both arms get the exact same speedups.
  Unsloth has **no `LoKr` (KronA) support** of its own — its `get_peft_model` only builds a
  `LoraConfig` — which is exactly why the adapters are attached manually here.

> **KRONA = KronA** = the Kronecker adapter (arXiv:2212.10650). In HuggingFace PEFT it is the
> **`LoKr`** adapter (`LoKrConfig`) — that is what we train.

Five code cells: install → load → data → train both → compare. That is it.


## Before you run

1. In Colab: **Runtime → Change runtime type → T4 GPU**.
2. Have a Hugging Face **read** token ready. The next cell reads the Colab secret
   `HF_TOKEN` if you stored one there, otherwise it prompts. The token is never saved.
3. Run the cells **top to bottom**.

Everything is a knob at the top of the config cell (`max_samples`, `steps`, `bs`, `accum`).
Raise them for a more meaningful comparison — it only costs time.


## 1. Install Unsloth & log in


In [ ]:
# 1) Install Unsloth + log in to Hugging Face
!pip install -q unsloth hf_transfer

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass
if not token:
    from getpass import getpass
    token = getpass("Hugging Face token (read): ")
login(token=token, add_to_git_credential=False)
print("logged in")


## 2. Load the base model

Unsloth loads the 4-bit weights and does the kbit preparation itself, so there is none of the
fp32 upcast / 8.75 GiB allocation that made the old `transformers` + `peft` version OOM on a T4.


In [ ]:
# 2) Config + load the base model with Unsloth
import os, gc, time, dataclasses, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # cuts 4-bit fragmentation

CFG = dict(
    model       = "unsloth/gemma-3-1b-it",   # small, text-only, Unsloth-supported
    dataset     = "databricks/databricks-dolly-15k",
    max_samples = 1000,     # rows used (raise for a longer run)
    eval_size   = 64,       # rows held out for evaluation
    max_seq     = 512,
    steps       = 60,       # training steps per method (identical for both)
    bs          = 2,
    accum       = 4,
    lr          = 2e-4,
    eval_every  = 20,
    r           = 16,       # same adapter rank/alpha for LoRA and KronA
    alpha       = 32,
    seed        = 42,
)
torch.manual_seed(CFG["seed"])

from unsloth import FastLanguageModel

def load_base():
    """Fresh 4-bit base model. Unsloth does the memory-safe kbit setup itself
    (no fp32 upcast, no 8.75 GiB allocation) + gradient checkpointing."""
    m, t = FastLanguageModel.from_pretrained(
        CFG["model"],
        load_in_4bit = True,
        max_seq_length = CFG["max_seq"],
        use_gradient_checkpointing = "unsloth",
        random_state = CFG["seed"],
    )
    m.config.use_cache = False
    return m, t

# smoke test: make sure the model loads before spending time on training
model, tok = load_base()
tok.padding_side = "right"
print(type(model).__name__, "| vocab", tok.vocab_size, "| pad", tok.pad_token)

del model                                   # cell 4 loads its own fresh copies
gc.collect(); torch.cuda.empty_cache()


## 3. Dataset


In [ ]:
# 3) Build the chat dataset (dolly-15k -> user/assistant turns)
from datasets import load_dataset

raw = (load_dataset(CFG["dataset"], split="train")
       .shuffle(seed=CFG["seed"])
       .select(range(CFG["max_samples"])))

def to_chat(e):
    user = e["instruction"].strip()
    if e.get("context"):
        user += "\n\n" + e["context"].strip()
    return {"messages": [{"role": "user", "content": user},
                         {"role": "assistant", "content": e["response"].strip()}]}

ds = (raw.filter(lambda e: e["response"] and e["response"].strip())
         .map(to_chat, remove_columns=raw.column_names))

cut = len(ds) - CFG["eval_size"]
train_ds, eval_ds = ds.select(range(cut)), ds.select(range(cut, len(ds)))
print(train_ds, eval_ds)


## 4. Train A (LoRA) and B (KronA)

Both methods share data, seed, steps, learning rate, batch size and target modules — only the
adapter type differs. Each run starts from a **fresh** Unsloth base model so they stay independent.


In [ ]:
# 4) Train A (LoRA) then B (KronA) — same data, seed, budget and target modules
from peft import LoraConfig, LoKrConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
PROMPTS = ["Explain what a Kronecker product is in one sentence.",
           "Give me 3 tips to speed up a PyTorch training loop.",
           "Write a haiku about fine-tuning a language model."]

def sft_config(**kw):
    """Keep only the kwargs this TRL version actually accepts."""
    ok = {f.name for f in dataclasses.fields(SFTConfig)}
    if "evaluation_strategy" in ok and "eval_strategy" not in ok:
        kw["evaluation_strategy"] = kw.pop("eval_strategy")
    return SFTConfig(**{k: v for k, v in kw.items() if k in ok})

def generate(m, prompts, n=100):
    FastLanguageModel.for_inference(m)
    out = []
    for p in prompts:
        ids = tok.apply_chat_template([{"role": "user", "content": p}],
                                      add_generation_prompt=True,
                                      return_tensors="pt", return_dict=False).to("cuda")
        g = m.generate(ids, max_new_tokens=n, do_sample=False, pad_token_id=tok.pad_token_id)
        out.append(tok.decode(g[0][ids.shape[-1]:], skip_special_tokens=True).strip())
    return out

def train(kind):
    m, _ = load_base()                       # fresh base -> the two runs stay independent
    shared = dict(r=CFG["r"], alpha=CFG["alpha"], target_modules=TARGETS,
                  bias="none", task_type="CAUSAL_LM")
    cfg = LoraConfig(lora_dropout=0.0, **shared) if kind == "lora" else LoKrConfig(**shared)
    m = get_peft_model(m, cfg)
    tp, total = m.get_nb_trainable_parameters()
    print(f"\n=== {kind.upper()} — {tp:,}/{total:,} trainable ({100 * tp / total:.3f}%) ===")

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    trainer = SFTTrainer(
        model = m, train_dataset = train_ds, eval_dataset = eval_ds, processing_class = tok,
        args = sft_config(
            output_dir = f"out/{kind}",
            max_steps = CFG["steps"],
            per_device_train_batch_size = CFG["bs"],
            gradient_accumulation_steps = CFG["accum"],
            learning_rate = CFG["lr"], lr_scheduler_type = "cosine", warmup_steps = 5,
            logging_steps = 5, eval_strategy = "steps", eval_steps = CFG["eval_every"],
            optim = "adamw_8bit", fp16 = True, seed = CFG["seed"], report_to = "none",
            max_length = CFG["max_seq"], packing = False,
        ))
    trainer.train()
    secs = time.time() - t0
    vram = torch.cuda.max_memory_reserved() / 1e9
    loss = float(trainer.evaluate()["eval_loss"])
    m.save_pretrained(f"out/adapter_{kind}")

    res = dict(params = tp, total = total, loss = loss, secs = secs, vram = vram,
               ppl = float(torch.exp(torch.tensor(loss))), samples = generate(m, PROMPTS))
    del trainer, m
    gc.collect(); torch.cuda.empty_cache()
    return res

RESULTS = {k: train(k) for k in ("lora", "krona")}


## 5. Compare & save


In [ ]:
# 5) Head-to-head table, samples, and the saved adapters
import pandas as pd
from IPython.display import display

tok.save_pretrained("out/adapter_lora")

display(pd.DataFrame([{
    "method":       k.upper(),
    "trainable":    r["params"],
    "trainable_%":  round(100 * r["params"] / r["total"], 3),
    "eval_loss":    round(r["loss"], 4),
    "perplexity":   round(r["ppl"], 2),
    "time_s":       round(r["secs"]),
    "peak_VRAM_GB": round(r["vram"], 2),
} for k, r in RESULTS.items()]))

for i, p in enumerate(PROMPTS):
    print("\n" + "-" * 70 + f"\nQ: {p}")
    for k, r in RESULTS.items():
        print(f"[{k.upper():5}] {r['samples'][i]}")

!mkdir -p out && cd out && zip -qr adapters.zip adapter_lora adapter_krona
print("\nsaved: out/adapter_lora, out/adapter_krona, out/adapters.zip")
